# DipDECK reference fits at initial K 15: raw ProtT5 and raw ESM-2 3B, seeds 0 to 9

**20 fits**, one per block: the full-record reference (all 1,577 GraphPart-retained proteins)
at initial K 15 instead of the paper's 35. Merge threshold 0.90, 100 pretraining and 50 clustering
epochs, the library loss (ClustPy `mean_squared_error`) and every check are those of the grouped
batch. It is a sensitivity run.

**Run:** upload this project folder to Google Drive as `MyDrive/explaining-plm-latent-spaces` (or set `PROJECT_DIR`), select **T4 GPU**, then **Run all**.
Setup, installation and environment checks are copied unchanged from the tested grouped notebooks,
so a runtime already set up by one of them is reused. The adapter and this run's small driver are embedded.

Outputs go to `DRIVE_ROOT/results/dipdeck_reference_k15/<block_id>/` and one ZIP per
fit to `DRIVE_ROOT/dipdeck_reference_k15_archives/`.
Completed blocks are validated and skipped on a rerun, so after an interruption run all again. A
stale `.running` file may be removed only after confirming its runtime has stopped.


In [ ]:
SEQUENCE = 1
BLOCK_ID = 'k15_01_prott5_raw1024_full_seed00'
PROJECT_DIR = "/content/drive/MyDrive/explaining-plm-latent-spaces"
CLUSTPY_COMMIT = '82548dc35a01d221474fbb44908b3621d4b97616'
EXPECTED_UPSTREAM_DIPDECK_SHA256 = 'c567d24c1722087a75c3eb56f17fe91152b6f818148083d908086b9fc78d438a'
EXPECTED_DIPDECK_SHA256 = 'cfbf629ab341bdbb5967af232c24cfc3f2e5fb09cd082fb28b317c45738b7420'
DATA_SHA256 = {'data/phosphatase/phosphatase/esm2_t36_3B_UR50D_full_length.h5': '00c27d5730ba59db9ca4178118a2ba5fc4e4e4ecc41266afdc9e000951e8571c', 'data/phosphatase/phosphatase/tmp/prot_t5.h5': '425e005d783d70c5ac9043ad2c7d794ce603a8ddbecffa405d569aeddb78c7c1'}
DRIVE_ROOT = PROJECT_DIR
EXPECTED_STANDALONE_SHA256 = '6e63002b5aa9d788baa61a1dd2295499e853a4050a6b3286092884727dcf70f5'
EXPECTED_ADAPTER_SHA256 = 'adedba78b4b6ba25d16f9ea9845f0b339e365d736626e3dd42ebc4ea55aa3bbd'
EXPECTED_DRIVER_SHA256 = '15557fe84c4649323b1df236d8375f34b4ef4aee74d904d696e72cdc127293e4'
REFERENCE_INITIAL_K = 15

BLOCKS = [(1, 'k15_01_prott5_raw1024_full_seed00'), (2, 'k15_02_prott5_raw1024_full_seed01'), (3, 'k15_03_prott5_raw1024_full_seed02'), (4, 'k15_04_prott5_raw1024_full_seed03'), (5, 'k15_05_prott5_raw1024_full_seed04'), (6, 'k15_06_prott5_raw1024_full_seed05'), (7, 'k15_07_prott5_raw1024_full_seed06'), (8, 'k15_08_prott5_raw1024_full_seed07'), (9, 'k15_09_prott5_raw1024_full_seed08'), (10, 'k15_10_prott5_raw1024_full_seed09'), (11, 'k15_11_esm2_3b_raw2560_full_seed00'), (12, 'k15_12_esm2_3b_raw2560_full_seed01'), (13, 'k15_13_esm2_3b_raw2560_full_seed02'), (14, 'k15_14_esm2_3b_raw2560_full_seed03'), (15, 'k15_15_esm2_3b_raw2560_full_seed04'), (16, 'k15_16_esm2_3b_raw2560_full_seed05'), (17, 'k15_17_esm2_3b_raw2560_full_seed06'), (18, 'k15_18_esm2_3b_raw2560_full_seed07'), (19, 'k15_19_esm2_3b_raw2560_full_seed08'), (20, 'k15_20_esm2_3b_raw2560_full_seed09')]


In [ ]:
from pathlib import Path
import json

setup_marker = Path("/content/dipdeck_library_loss_setup.json")
setup_key = {
    "project": PROJECT_DIR,
    "clustpy": CLUSTPY_COMMIT,
    "adapter": EXPECTED_ADAPTER_SHA256,
}
REUSE_SETUP = setup_marker.exists()
if REUSE_SETUP and json.loads(setup_marker.read_text()) != setup_key:
    raise RuntimeError("This runtime belongs to another notebook build. Start a fresh runtime.")
if REUSE_SETUP:
    from google.colab import drive
    drive.mount("/content/drive")
    import hashlib
    import os
    import subprocess
    repo = Path("/content/project").resolve()
    clustpy_source = Path("/content/ClustPy").resolve()
    drive_root = Path(DRIVE_ROOT)
    repo_commit = "project-folder"
    for relative, expected in DATA_SHA256.items():
        if hashlib.sha256((repo / relative).read_bytes()).hexdigest() != expected:
            raise RuntimeError(f"Cached input changed: {relative}")
    print("Reusing this runtime's setup; all version and source checks still run.")
else:
    from google.colab import drive
    drive.mount("/content/drive")

    from pathlib import Path
    import hashlib
    import os
    import shutil
    import subprocess

    repo = Path("/content/project").resolve()
    clustpy_source = Path("/content/ClustPy").resolve()
    drive_root = Path(DRIVE_ROOT)
    drive_root.mkdir(parents=True, exist_ok=True)

    for target in (repo, clustpy_source):
        if target.parent != Path("/content"):
            raise RuntimeError(f"Refusing to replace unexpected path: {target}")
        if target.exists():
            shutil.rmtree(target)

    project_dir = Path(PROJECT_DIR)
    if not (project_dir / "src/algo/dipdeck/dipdeck_standalone.py").is_file():
        raise FileNotFoundError(f"Upload this project folder to Google Drive at {project_dir}, then rerun.")
    for part in ("src", "data", "results/graphpart", "results/ec_audit"):
        shutil.copytree(project_dir / part, repo / part)
    subprocess.run(["git", "clone", "https://github.com/collinleiber/ClustPy.git", str(clustpy_source)], check=True)
    subprocess.run(["git", "-C", str(clustpy_source), "checkout", CLUSTPY_COMMIT], check=True)

    dipdeck_source = clustpy_source / "clustpy/deep/dipdeck.py"
    source_text = dipdeck_source.read_text(encoding="utf-8")
    upstream_sha256 = hashlib.sha256(source_text.encode()).hexdigest()
    if upstream_sha256 != EXPECTED_UPSTREAM_DIPDECK_SHA256:
        raise RuntimeError(
            f"Unexpected upstream DipDECK source: expected {EXPECTED_UPSTREAM_DIPDECK_SHA256}, "
            f"got {upstream_sha256}"
        )
    for old, new in (
        ("self.labels.to(device)", "self.labels = self.labels.to(device)"),
        ("self.cluster_centers.to(device)", "self.cluster_centers = self.cluster_centers.to(device)"),
        ("self.dip_matrix.to(device)", "self.dip_matrix = self.dip_matrix.to(device)"),
    ):
        if source_text.count(old) != 1:
            raise RuntimeError(f"CUDA patch target is absent or ambiguous: {old}")
        source_text = source_text.replace(old, new)
    dipdeck_source.write_text(source_text, encoding="utf-8")
    patched_sha256 = hashlib.sha256(dipdeck_source.read_bytes()).hexdigest()
    if patched_sha256 != EXPECTED_DIPDECK_SHA256:
        raise RuntimeError(
            f"Patched DipDECK source mismatch: expected {EXPECTED_DIPDECK_SHA256}, got {patched_sha256}"
        )

    repo_commit = "project-folder"

    for relative, expected in DATA_SHA256.items():
        path = repo / relative
        if not path.is_file():
            raise FileNotFoundError(f"Project copy is missing {relative}")
        observed = hashlib.sha256(path.read_bytes()).hexdigest()
        if observed != expected:
            raise RuntimeError(f"{relative}: expected SHA256 {expected}, got {observed}")
    if (repo / "data/phosphatase/phosphatase/data_standard.parquetbundle").is_file() is False:
        raise FileNotFoundError("Project copy is missing the phosphatase annotation bundle")

    os.chdir(repo)
    print(f"Repository: {repo}")
    print(f"Project copied from: {PROJECT_DIR}")
    print(f"Data checked: {list(DATA_SHA256)}")
    print(f"ClustPy revision: {CLUSTPY_COMMIT}")


In [ ]:
if REUSE_SETUP:
    COLAB_PINS = {'Cython': '3.3.0', 'numpy': '2.1.3', 'scipy': '1.18.1', 'scikit-learn': '1.6.1', 'pandas': '2.2.3', 'h5py': '3.16.0', 'pyarrow': '25.0.1', 'tqdm': '4.70.0', 'pytest': '9.1.1', 'psutil': '7.2.2', 'torch': '2.11.0+cu128', 'torchvision': '0.26.0+cu128'}
    print('Dependencies already installed; verifying them in the next cell.')
else:
    COLAB_PINS = {
        "Cython": "3.3.0",
        "numpy": "2.1.3",
        "scipy": "1.18.1",
        "scikit-learn": "1.6.1",
        "pandas": "2.2.3",
        "h5py": "3.16.0",
        "pyarrow": "25.0.1",
        "tqdm": "4.70.0",
        "pytest": "9.1.1",
        "psutil": "7.2.2",
        "torch": "2.11.0+cu128",
        "torchvision": "0.26.0+cu128",
    }

    import os
    import subprocess
    import sys

    os.chdir("/content")
    install_names = [
        "Cython", "numpy", "scipy", "scikit-learn", "pandas",
        "h5py", "pyarrow", "tqdm", "pytest", "psutil",
    ]
    requirements = [f"{name}=={COLAB_PINS[name]}" for name in install_names]
    install_result = subprocess.run(
        [sys.executable, "-m", "pip", "install", *requirements],
        check=True, capture_output=True, text=True,
    )
    print(install_result.stdout)
    if install_result.stderr:
        print(install_result.stderr, file=sys.stderr)

    if "torch" in sys.modules or "torchvision" in sys.modules:
        raise RuntimeError("torch is already imported. Use Runtime > Restart session, then run all cells.")
    torch_result = subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            f"torch=={COLAB_PINS['torch']}", f"torchvision=={COLAB_PINS['torchvision']}",
            "--index-url", "https://download.pytorch.org/whl/cu128",
        ],
        check=True, capture_output=True, text=True,
    )
    print(torch_result.stdout[-2000:])
    if torch_result.stderr:
        print(torch_result.stderr[-2000:], file=sys.stderr)

    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "--no-deps",
        "--force-reinstall", "--no-cache-dir", "--editable", str(clustpy_source),
    ] , check=True)

    pip_check = subprocess.run(
        [sys.executable, "-m", "pip", "check"], capture_output=True, text=True,
    )
    print(pip_check.stdout)
    if pip_check.stderr:
        print(pip_check.stderr, file=sys.stderr)
    if pip_check.returncode != 0:
        print("pip check reported the Colab base-environment warning above.", file=sys.stderr)


In [ ]:
import hashlib
import importlib
import importlib.metadata as metadata
import json
import os
import platform
import sys
from pathlib import Path

for module_name in (
    "Cython", "h5py", "pandas", "psutil", "pyarrow", "pytest", "scipy", "sklearn", "tqdm",
):
    importlib.import_module(module_name)

import numpy as np
import torch
import torchvision
from threadpoolctl import threadpool_info

if not (clustpy_source / "clustpy/__init__.py").is_file():
    raise FileNotFoundError(f"ClustPy package is missing from the verified checkout: {clustpy_source}")
sys.path.insert(0, str(clustpy_source))
import clustpy
from clustpy.deep import DipDECK
if Path(clustpy.__file__).resolve().parent != (clustpy_source / "clustpy").resolve():
    raise RuntimeError(f"Imported ClustPy is not the verified checkout: {clustpy.__file__}")

observed_versions = {name: metadata.version(name) for name in COLAB_PINS}
version_mismatches = {
    name: {"expected": expected, "observed": observed_versions[name]}
    for name, expected in COLAB_PINS.items()
    if observed_versions[name] != expected
}
if version_mismatches:
    raise RuntimeError(f"Colab dependency mismatch: {version_mismatches}")

dipdeck_source = clustpy_source / "clustpy/deep/dipdeck.py"
dipdeck_sha256 = hashlib.sha256(dipdeck_source.read_bytes()).hexdigest()
if dipdeck_sha256 != EXPECTED_DIPDECK_SHA256:
    raise RuntimeError(
        f"DipDECK source mismatch: expected {EXPECTED_DIPDECK_SHA256}, got {dipdeck_sha256}"
    )

environment = {
    "python": platform.python_version(),
    "project_commit": repo_commit,
    "clustpy_version": metadata.version("clustpy"),
    "clustpy_path": clustpy.__file__,
    "clustpy_commit": CLUSTPY_COMMIT,
    "dipdeck_sha256": dipdeck_sha256,
    "package_versions": observed_versions,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device_count": torch.cuda.device_count(),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "torch_num_threads": torch.get_num_threads(),
    "torch_num_interop_threads": torch.get_num_interop_threads(),
    "thread_environment": {
        name: os.environ.get(name)
        for name in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS")
    },
    "threadpool_info": threadpool_info(),
}
print(json.dumps(environment, indent=2))
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU. In Colab choose Runtime > Change runtime type > GPU, then run all again.")
if "T4" not in torch.cuda.get_device_name(0):
    raise RuntimeError("This batch is pinned to T4. Select a T4 runtime; do not substitute silently.")
if hashlib.sha256((repo / "src/algo/dipdeck/dipdeck_standalone.py").read_bytes()).hexdigest() != EXPECTED_STANDALONE_SHA256:
    raise RuntimeError("Pinned standalone source differs from the tested runner.")
setup_marker.write_text(json.dumps(setup_key), encoding="utf-8")


In [ ]:
import hashlib
import subprocess
import sys
from pathlib import Path

adapter_source = 'from __future__ import annotations\r\n\r\nimport argparse\r\nfrom contextlib import contextmanager\r\nfrom functools import partial\r\nimport hashlib\r\nimport importlib.util\r\nimport inspect\r\nimport json\r\nfrom pathlib import Path\r\nimport shutil\r\nimport subprocess\r\nimport sys\r\nfrom types import ModuleType\r\nfrom typing import Any, Iterator\r\nimport zipfile\r\n\r\n\r\nPROJECT_COMMIT = "project-folder"\r\nSTANDALONE_SHA256 = "6e63002b5aa9d788baa61a1dd2295499e853a4050a6b3286092884727dcf70f5"\r\nLOSS = {\r\n    "ssl_loss": "mean_squared_error",\r\n    "ssl_loss_reduction": "sum_features_mean_samples",\r\n    "ssl_loss_weight": 1.0,\r\n    "clustering_loss_weight": 1.0,\r\n}\r\nARMS = (\r\n    ("prott5_raw1024_full", "prott5", None, False),\r\n    ("esm2_3b_raw2560_full", "esm2_3b", None, False),\r\n    ("prott5_pca67_full", "prott5", 67, False),\r\n    ("esm2_3b_pca45_full", "esm2_3b", 45, False),\r\n    ("prott5_pca113_full", "prott5", 113, False),\r\n    ("prott5_pca34_full", "prott5", 34, False),\r\n    ("prott5_raw1024_dedup", "prott5", None, True),\r\n    ("prott5_pca67_dedup", "prott5", 67, True),\r\n    ("prott5_pca113_dedup", "prott5", 113, True),\r\n    ("prott5_pca34_dedup", "prott5", 34, True),\r\n)\r\n\r\n\r\ndef block_spec(sequence: int) -> dict[str, Any]:\r\n    if not 1 <= sequence <= len(ARMS) * 10:\r\n        raise ValueError("sequence must be between 1 and 100")\r\n    arm, embedding, pca, dedup = ARMS[(sequence - 1) // 10]\r\n    seed = (sequence - 1) % 10\r\n    return {\r\n        "sequence": sequence, "arm": arm, "embedding": embedding,\r\n        "pca_dimensions": pca, "dedup": dedup, "seed": seed,\r\n        "block_id": f"{sequence:03d}_{arm}_seed{seed:02d}",\r\n        "grid_fits": 100, "reference_fits": 1,\r\n    }\r\n\r\n\r\ndef digest(path: Path) -> str:\r\n    return hashlib.sha256(path.read_bytes()).hexdigest()\r\n\r\n\r\ndef write_json(path: Path, value: Any) -> None:\r\n    temporary = path.with_name(path.name + ".partial")\r\n    temporary.write_text(json.dumps(value, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\r\n    temporary.replace(path)\r\n\r\n\r\ndef load_standalone(repo: Path, *, require_commit: bool) -> ModuleType:\r\n    path = repo / "src" / "algo" / "dipdeck" / "dipdeck_standalone.py"\r\n    if digest(path) != STANDALONE_SHA256:\r\n        raise ValueError("DipDECK runner differs from the tested source; do not bypass this check")\r\n    spec = importlib.util.spec_from_file_location("dipdeck_library_loss_base", path)\r\n    if spec is None or spec.loader is None:\r\n        raise ImportError(f"Cannot load {path}")\r\n    module = importlib.util.module_from_spec(spec)\r\n    sys.modules[spec.name] = module\r\n    spec.loader.exec_module(module)\r\n    return module\r\n\r\n\r\ndef make_factory(ds: ModuleType):\r\n    import torch\r\n    from clustpy.deep import DipDECK\r\n    from clustpy.deep._utils import mean_squared_error\r\n\r\n    if inspect.signature(DipDECK.__init__).parameters["ssl_loss_fn"].default is not mean_squared_error:\r\n        raise ValueError("Pinned DipDECK no longer defaults to the expected library loss")\r\n    calls = []\r\n\r\n    def factory(config, device, source_root):\r\n        estimator, runtime = ds._create_dipdeck(config, device, source_root)\r\n        if not isinstance(estimator.ssl_loss_fn, torch.nn.MSELoss) or estimator.ssl_loss_fn.reduction != "mean":\r\n            raise ValueError("Original factory no longer supplies MSELoss(mean)")\r\n        estimator.ssl_loss_fn = mean_squared_error\r\n        calls.append(config)\r\n        print(\r\n            f"New-loss fit {len(calls)}: seed={config.seed}, "\r\n            f"K={config.n_clusters_init}, threshold={config.dip_merge_threshold}",\r\n            flush=True,\r\n        )\r\n        return estimator, runtime\r\n\r\n    return factory, calls\r\n\r\n\r\n@contextmanager\r\ndef reference_loss_reporting(ds: ModuleType) -> Iterator[None]:\r\n    original = ds._persist_and_reopen\r\n\r\n    def persist(paths, row, run_row, cluster_rows, payload):\r\n        run_row.update(LOSS)\r\n        return original(paths, row, run_row, cluster_rows, payload)\r\n\r\n    ds._persist_and_reopen = persist\r\n    try:\r\n        yield\r\n    finally:\r\n        ds._persist_and_reopen = original\r\n\r\n\r\ndef options(block: dict[str, Any]) -> dict[str, Any]:\r\n    return {\r\n        "embedding": block["embedding"], "dedup": block["dedup"],\r\n        "pca_dimensions": block["pca_dimensions"], "seeds": (block["seed"],),\r\n    }\r\n\r\n\r\ndef owned_paths(ds: ModuleType, root: Path, block: dict[str, Any], kind: str):\r\n    space = ds._input_space(block["embedding"], block["pca_dimensions"])\r\n    if kind == "reference":\r\n        return ds._scientific_reference_paths(\r\n            root, block["seed"], dedup=block["dedup"], input_space=space\r\n        )\r\n    return ds._scientific_paths(root, dedup=block["dedup"], input_space=space)\r\n\r\n\r\ndef identity(block: dict[str, Any]) -> dict[str, Any]:\r\n    return {\r\n        **block, **LOSS, "project_commit": PROJECT_COMMIT,\r\n        "standalone_sha256": STANDALONE_SHA256, "adapter_sha256": digest(Path(__file__)),\r\n    }\r\n\r\n\r\ndef check_identity(root: Path, block: dict[str, Any], *, create: bool) -> None:\r\n    expected = identity(block)\r\n    path = root / "block.json"\r\n    if path.exists():\r\n        if json.loads(path.read_text(encoding="utf-8")) != expected:\r\n            raise ValueError(f"Different block, loss or adapter already occupies {root}")\r\n    elif create:\r\n        if root.exists() and any(root.iterdir()):\r\n            raise ValueError(f"Nonempty output without block provenance: {root}")\r\n        root.mkdir(parents=True, exist_ok=True)\r\n        write_json(path, expected)\r\n        shutil.copyfile(__file__, root / "dipdeck_library_loss.py")\r\n    else:\r\n        raise FileNotFoundError(path)\r\n\r\n\r\ndef validate_stage(ds: ModuleType, root: Path, block: dict[str, Any], kind: str):\r\n    kwargs = options(block)\r\n    if kind == "reference":\r\n        runs = ds._validate_scientific_reference_outputs(root, **kwargs)\r\n        for key, value in LOSS.items():\r\n            if not (runs[key] == value).all():\r\n                raise ValueError(f"Reference reports the wrong {key}")\r\n    else:\r\n        runs = ds._validate_scientific_outputs(\r\n            owned_paths(ds, root, block, kind), expected_rows=100, **kwargs\r\n        )\r\n    if len(runs) != (1 if kind == "reference" else 100):\r\n        raise ValueError(f"Wrong number of {kind} fits")\r\n    return runs\r\n\r\n\r\ndef check_completion(ds: ModuleType, root: Path, block: dict[str, Any], kind: str):\r\n    import pandas as pd\r\n\r\n    check_identity(root, block, create=False)\r\n    marker = json.loads((root / f"{kind}.complete.json").read_text(encoding="utf-8"))\r\n    if marker["block_id"] != block["block_id"] or marker["kind"] != kind:\r\n        raise ValueError("Completion marker belongs to another block")\r\n    for relative, expected in marker["sha256"].items():\r\n        path = (root / relative).resolve()\r\n        if not path.is_relative_to(root.resolve()) or digest(path) != expected:\r\n            raise ValueError(f"Completed output changed: {relative}")\r\n    runs = validate_stage(ds, root, block, kind)\r\n    provenance = pd.read_csv(root / f"{kind}_loss.csv")\r\n    if provenance["row"].tolist() != runs["row"].tolist():\r\n        raise ValueError("Loss provenance does not cover exactly the validated rows")\r\n    for key, value in LOSS.items():\r\n        if not (provenance[key] == value).all():\r\n            raise ValueError(f"Wrong {key} in loss provenance")\r\n    return runs\r\n\r\n\r\ndef complete_stage(ds: ModuleType, root: Path, block: dict[str, Any], kind: str, runs) -> None:\r\n    provenance = runs[["row"]].copy()\r\n    for key, value in LOSS.items():\r\n        provenance[key] = value\r\n    provenance.to_csv(root / f"{kind}_loss.csv", index=False)\r\n    files = list(owned_paths(ds, root, block, kind).root.rglob("*"))\r\n    files += [root / "block.json", root / "dipdeck_library_loss.py",\r\n              root / f"{kind}_loss.csv", root / f"{kind}_environment.json"]\r\n    write_json(root / f"{kind}.complete.json", {\r\n        "block_id": block["block_id"], "kind": kind, "fits": len(runs),\r\n        "sha256": {path.relative_to(root).as_posix(): digest(path)\r\n                   for path in sorted(files) if path.is_file()},\r\n    })\r\n\r\n\r\ndef package(ds: ModuleType, root: Path, block: dict[str, Any], kind: str, archives: Path) -> Path:\r\n    kinds = ("reference",) if kind == "reference" else ("reference", "grid")\r\n    files = {root / "block.json", root / "dipdeck_library_loss.py"}\r\n    for part in kinds:\r\n        check_completion(ds, root, block, part)\r\n        files.update(path for path in owned_paths(ds, root, block, part).root.rglob("*") if path.is_file())\r\n        files.update(root / name for name in (\r\n            f"{part}.complete.json", f"{part}_loss.csv", f"{part}_environment.json"\r\n        ))\r\n    archives.mkdir(parents=True, exist_ok=True)\r\n    suffix = "reference" if kind == "reference" else "complete"\r\n    target = archives / f"{block[\'block_id\']}_{suffix}.zip"\r\n    temporary = target.with_suffix(".partial.zip")\r\n    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED) as packed:\r\n        for path in sorted(files):\r\n            packed.write(path, (Path(root.name) / path.relative_to(root)).as_posix())\r\n    with zipfile.ZipFile(temporary) as packed:\r\n        if packed.testzip() is not None or len(packed.namelist()) != len(files):\r\n            raise ValueError(f"Archive failed validation: {temporary}")\r\n    temporary.replace(target)\r\n    print(f"Output archive: {target}", flush=True)\r\n    print(f"Output SHA256: {digest(target)}", flush=True)\r\n    return target\r\n\r\n\r\ndef run_stage(ds: ModuleType, root: Path, block: dict[str, Any], kind: str, source_root: Path) -> None:\r\n    import torch\r\n\r\n    factory, calls = make_factory(ds)\r\n    if not torch.cuda.is_available() or "T4" not in torch.cuda.get_device_name(0):\r\n        raise RuntimeError("Use a T4 runtime for this batch; no automatic hardware substitution")\r\n    write_json(root / f"{kind}_environment.json", {\r\n        "python": sys.version, "torch": torch.__version__,\r\n        "cuda_device": torch.cuda.get_device_name(0),\r\n        "clustpy_commit": ds.CLUSTPY_COMMIT, "dipdeck_sha256": ds.DIPDECK_SOURCE_SHA256,\r\n    })\r\n    kwargs = options(block)\r\n    with reference_loss_reporting(ds):\r\n        run = ds._run_scientific_reference if kind == "reference" else ds._run_scientific_grid\r\n        run(root, device="cuda", source_root=source_root, estimator_factory=factory, **kwargs)\r\n    expected = 1 if kind == "reference" else 100\r\n    if len(calls) != expected:\r\n        raise ValueError(f"Expected {expected} new-loss fits, constructed {len(calls)}")\r\n    runs = validate_stage(ds, root, block, kind)\r\n    complete_stage(ds, root, block, kind, runs)\r\n    print(f"VALIDATED {block[\'block_id\']}: {kind}, {len(runs)} fits", flush=True)\r\n\r\n\r\ndef execute(ds: ModuleType, base: Path, archives: Path, sequence: int, kind: str,\r\n            source_root: Path, *, validate_only: bool = False) -> None:\r\n    if base.name != "dipdeck_library_loss":\r\n        raise ValueError("Output base must be named dipdeck_library_loss to isolate the old results")\r\n    block = block_spec(sequence)\r\n    root = base / block["block_id"]\r\n    check_identity(root, block, create=not validate_only)\r\n    lock = root / ".running"\r\n    try:\r\n        handle = lock.open("x", encoding="utf-8")\r\n    except FileExistsError as error:\r\n        raise RuntimeError(\r\n            f"{lock} exists. Do not run this block twice. If its runtime has stopped, "\r\n            "remove only this lock file and rerun."\r\n        ) from error\r\n    try:\r\n        with handle:\r\n            handle.write(f"{kind}\\n")\r\n        if kind == "grid":\r\n            check_completion(ds, root, block, "reference")\r\n        marker = root / f"{kind}.complete.json"\r\n        if marker.exists() or validate_only:\r\n            check_completion(ds, root, block, kind)\r\n            print(f"SKIP {block[\'block_id\']} {kind}: completed output validates", flush=True)\r\n        else:\r\n            print(f"RUN {block[\'block_id\']} {kind}; incomplete output of this stage is replaced", flush=True)\r\n            run_stage(ds, root, block, kind, source_root)\r\n        package(ds, root, block, kind, archives)\r\n    finally:\r\n        lock.unlink()\r\n\r\n\r\ndef merge_arm(ds: ModuleType, base: Path, destination: Path, arm: str) -> None:\r\n    import pandas as pd\r\n\r\n    if base.name != "dipdeck_library_loss" or destination.name != "dipdeck_library_loss_merged":\r\n        raise ValueError("Use dedicated dipdeck_library_loss and dipdeck_library_loss_merged directories")\r\n    if destination == base or destination.is_relative_to(base) or base.is_relative_to(destination):\r\n        raise ValueError("Merged output must be separate from the block source tree")\r\n    blocks = [block_spec(i) for i in range(1, 101) if block_spec(i)["arm"] == arm]\r\n    sources = [base / block["block_id"] for block in blocks]\r\n    for root, block in zip(sources, blocks):\r\n        for kind in ("reference", "grid"):\r\n            check_completion(ds, root, block, kind)\r\n    marker = destination / "library_loss.json"\r\n    expected = {**LOSS, "adapter_sha256": digest(Path(__file__)),\r\n                "project_commit": PROJECT_COMMIT, "standalone_sha256": STANDALONE_SHA256}\r\n    if marker.exists():\r\n        if json.loads(marker.read_text(encoding="utf-8")) != expected:\r\n            raise ValueError("Merged root belongs to another loss or adapter")\r\n    elif destination.exists() and any(destination.iterdir()):\r\n        raise ValueError("Refusing to merge into an existing tree without new-loss provenance")\r\n    destination.mkdir(parents=True, exist_ok=True)\r\n    write_json(marker, expected)\r\n    for suffix in ("loss.csv", "blocks.json"):\r\n        (destination / f"{arm}_{suffix}").unlink(missing_ok=True)\r\n    bound = {}\r\n    if blocks[0]["dedup"]:\r\n        for name in ("_scientific_paths", "_scientific_reference_paths",\r\n                     "_validate_scientific_outputs", "_validate_scientific_reference_outputs"):\r\n            bound[name] = getattr(ds, name)\r\n            setattr(ds, name, partial(bound[name], dedup=True))\r\n    try:\r\n        ds._merge_seed_halves(\r\n            sources, destination, embedding=blocks[0]["embedding"],\r\n            pca_dimensions=blocks[0]["pca_dimensions"],\r\n        )\r\n    finally:\r\n        for name, function in bound.items():\r\n            setattr(ds, name, function)\r\n    destination.mkdir(parents=True, exist_ok=True)\r\n    provenance = pd.concat(\r\n        [pd.read_csv(root / f"{kind}_loss.csv") for root in sources for kind in ("reference", "grid")],\r\n        ignore_index=True,\r\n    )\r\n    if len(provenance) != 1010 or provenance["row"].duplicated().any():\r\n        raise ValueError("Merged arm does not contain 1,010 distinct fits")\r\n    provenance.to_csv(destination / f"{arm}_loss.csv", index=False)\r\n    write_json(destination / f"{arm}_blocks.json", [identity(block) for block in blocks])\r\n    print(f"MERGED and validated {arm}: 1,000 fold fits and 10 references", flush=True)\r\n\r\n\r\ndef main() -> None:\r\n    parser = argparse.ArgumentParser(description=__doc__)\r\n    parser.add_argument("--repo", type=Path, required=True)\r\n    parser.add_argument("--output-base", type=Path, required=True)\r\n    parser.add_argument("--sequence", type=int)\r\n    parser.add_argument("--kind", choices=("reference", "grid"))\r\n    parser.add_argument("--source-root", type=Path)\r\n    parser.add_argument("--archives", type=Path)\r\n    parser.add_argument("--validate-only", action="store_true")\r\n    parser.add_argument("--merge-arm", choices=[arm[0] for arm in ARMS])\r\n    parser.add_argument("--merge-output", type=Path)\r\n    args = parser.parse_args()\r\n    if args.merge_arm:\r\n        if args.merge_output is None:\r\n            parser.error("--merge-arm requires --merge-output")\r\n        ds = load_standalone(args.repo.resolve(), require_commit=False)\r\n        merge_arm(ds, args.output_base.resolve(), args.merge_output.resolve(), args.merge_arm)\r\n    else:\r\n        if args.sequence is None or args.kind is None or args.archives is None or args.source_root is None:\r\n            parser.error("running a block requires --sequence, --kind, --archives and --source-root")\r\n        sys.path.insert(0, str(args.source_root.resolve()))\r\n        ds = load_standalone(args.repo.resolve(), require_commit=not args.validate_only)\r\n        execute(ds, args.output_base.resolve(), args.archives.resolve(), args.sequence,\r\n                args.kind, args.source_root.resolve(), validate_only=args.validate_only)\r\n\r\n\r\nif __name__ == "__main__":\r\n    main()\r\n'
driver_source = 'from __future__ import annotations\n\nimport argparse\nimport hashlib\nimport importlib.util\nfrom pathlib import Path\nimport sys\nfrom types import ModuleType\nfrom typing import Any\n\nADAPTER_SHA256 = "adedba78b4b6ba25d16f9ea9845f0b339e365d736626e3dd42ebc4ea55aa3bbd"\nREFERENCE_INITIAL_K = 15\nREFERENCE_THRESHOLD = 0.90\nARMS = (\n    ("prott5_raw1024_full", "prott5", None, False),\n    ("esm2_3b_raw2560_full", "esm2_3b", None, False),\n)\nSEEDS = tuple(range(10))\nOUTPUT_BASE_NAME = "dipdeck_reference_k15"\n\n\ndef digest(path: Path) -> str:\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef load_adapter(path: Path) -> ModuleType:\n    if digest(path) != ADAPTER_SHA256:\n        raise ValueError("The library-loss adapter differs from the tested one; do not bypass this check")\n    spec = importlib.util.spec_from_file_location("dipdeck_library_loss", path)\n    if spec is None or spec.loader is None:\n        raise ImportError(f"Cannot load {path}")\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    return module\n\n\ndef block_spec(sequence: int) -> dict[str, Any]:\n    if not 1 <= sequence <= len(ARMS) * len(SEEDS):\n        raise ValueError(f"sequence must be between 1 and {len(ARMS) * len(SEEDS)}")\n    arm, embedding, pca, dedup = ARMS[(sequence - 1) // len(SEEDS)]\n    seed = SEEDS[(sequence - 1) % len(SEEDS)]\n    return {\n        "sequence": sequence, "arm": arm, "embedding": embedding,\n        "pca_dimensions": pca, "dedup": dedup, "seed": seed,\n        "block_id": f"k15_{sequence:02d}_{arm}_seed{seed:02d}",\n        "grid_fits": 0, "reference_fits": 1,\n        "reference_initial_k": REFERENCE_INITIAL_K, "driver_sha256": driver_sha256(),\n    }\n\n\ndef driver_sha256() -> str:\n    return hashlib.sha256(Path(__file__).read_bytes().replace(b"\\r\\n", b"\\n")).hexdigest()\n\n\ndef set_reference_k(ds: ModuleType) -> ModuleType:\n    ds.SCIENTIFIC_REFERENCE_INITIAL_CLUSTERS = REFERENCE_INITIAL_K\n    configs = ds._scientific_reference_configs(SEEDS)\n    if {(c.n_clusters_init, c.dip_merge_threshold, c.pretrain_epochs, c.clustering_epochs)\n            for c in configs} != {(REFERENCE_INITIAL_K, REFERENCE_THRESHOLD, 100, 50)}:\n        raise ValueError("The reference configuration is not initial K 15, threshold 0.90, 100/50 epochs")\n    return ds\n\n\ndef execute(adapter: ModuleType, ds: ModuleType, base: Path, archives: Path, sequence: int,\n            source_root: Path, *, validate_only: bool = False) -> Path:\n    if base.name != OUTPUT_BASE_NAME:\n        raise ValueError(f"Output base must be named {OUTPUT_BASE_NAME} to keep it apart from the tested batch")\n    if ds.SCIENTIFIC_REFERENCE_INITIAL_CLUSTERS != REFERENCE_INITIAL_K:\n        raise ValueError("Call set_reference_k first")\n    block = block_spec(sequence)\n    root = base / block["block_id"]\n    adapter.check_identity(root, block, create=not validate_only)\n    lock = root / ".running"\n    try:\n        handle = lock.open("x", encoding="utf-8")\n    except FileExistsError as error:\n        raise RuntimeError(\n            f"{lock} exists. Do not run this block twice. If its runtime has stopped, "\n            "remove only this lock file and rerun."\n        ) from error\n    try:\n        with handle:\n            handle.write("reference\\n")\n        if (root / "reference.complete.json").exists() or validate_only:\n            adapter.check_completion(ds, root, block, "reference")\n            print(f"SKIP {block[\'block_id\']} reference: completed output validates", flush=True)\n        else:\n            print(f"RUN {block[\'block_id\']} reference at initial K {REFERENCE_INITIAL_K}", flush=True)\n            adapter.run_stage(ds, root, block, "reference", source_root)\n        return adapter.package(ds, root, block, "reference", archives)\n    finally:\n        lock.unlink()\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=(__doc__ or "").partition("\\n")[0])\n    parser.add_argument("--adapter", type=Path, required=True)\n    parser.add_argument("--repo", type=Path, required=True)\n    parser.add_argument("--output-base", type=Path, required=True)\n    parser.add_argument("--archives", type=Path, required=True)\n    parser.add_argument("--source-root", type=Path, required=True)\n    parser.add_argument("--sequence", type=int, required=True)\n    parser.add_argument("--validate-only", action="store_true")\n    args = parser.parse_args()\n    sys.path.insert(0, str(args.source_root.resolve()))\n    adapter = load_adapter(args.adapter.resolve())\n    ds = set_reference_k(adapter.load_standalone(args.repo.resolve(), require_commit=not args.validate_only))\n    execute(adapter, ds, args.output_base.resolve(), args.archives.resolve(), args.sequence,\n            args.source_root.resolve(), validate_only=args.validate_only)\n\n\nif __name__ == "__main__":\n    main()\n'
for name, text, expected in (('adapter', adapter_source, EXPECTED_ADAPTER_SHA256),
                             ('driver', driver_source, EXPECTED_DRIVER_SHA256)):
    if hashlib.sha256(text.encode('utf-8')).hexdigest() != expected:
        raise RuntimeError(f'Embedded {name} is damaged')
adapter_path = Path('/content/dipdeck_library_loss_runner.py')
driver_path = Path('/content/dipdeck_reference_k15_runner.py')
adapter_path.write_bytes(adapter_source.encode('utf-8'))
driver_path.write_bytes(driver_source.encode('utf-8'))
output_base = drive_root / 'results' / 'dipdeck_reference_k15'
archive_root = drive_root / 'dipdeck_reference_k15_archives'
common_command = [
    sys.executable, '-u', str(driver_path), '--adapter', str(adapter_path), '--repo', str(repo),
    '--output-base', str(output_base), '--archives', str(archive_root),
    '--source-root', str(clustpy_source), '--sequence', str(SEQUENCE),
]
print(f'Reference fits at initial K {REFERENCE_INITIAL_K}; loss: mean_squared_error (sum features, mean samples)')
print(f'Output isolation: {output_base}')


In [ ]:
sequence_index = common_command.index('--sequence') + 1
for block_sequence, block_id in BLOCKS:
    command = common_command.copy()
    command[sequence_index] = str(block_sequence)
    print(f'Starting {block_id}', flush=True)
    subprocess.run(command, cwd=repo, check=True)
    print(f'Completed {block_id}; reference ZIP is in {archive_root}', flush=True)
print('All 20 reference fits validated and archived.', flush=True)


Success means **All 20 reference fits validated and archived** was printed, with one ZIP per fit. Download the archive folder for local validation. A failed cell means the run is unfinished; completed ZIPs remain usable.
